<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W6D2__DailyChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# Daily Challenge: Multi-Attention & Transformer Comparisons
# One Cell Solution
# ==========================================================

# =========================
# Imports
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# =========================
# 1. Single-Head Attention
# =========================

class Attention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / (Q.size(-1) ** 0.5)

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output, attention_weights


# =========================
# Dummy Example
# =========================

batch_size = 2
seq_len = 8
hidden_dim = 64

x = torch.randn(
    batch_size,
    seq_len,
    hidden_dim
)

single_attention = Attention(hidden_dim)

output, attention_weights = single_attention(x)

print("===== SINGLE HEAD =====")
print("Input Shape :", x.shape)
print("Output Shape:", output.shape)
print("Attention Shape:", attention_weights.shape)

# =========================
# Visualize Attention
# =========================

plt.figure(figsize=(6,5))
plt.imshow(
    attention_weights[0].detach().numpy(),
    cmap="Blues"
)
plt.title("Single Head Attention")
plt.colorbar()
plt.show()

# =========================
# 2. Multi-Head Attention
# =========================

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        hidden_dim,
        num_heads,
        dropout=0.1
    ):
        super().__init__()

        assert hidden_dim % num_heads == 0

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        batch_size = x.shape[0]

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.reshape(
            batch_size,
            -1,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        K = K.reshape(
            batch_size,
            -1,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        V = V.reshape(
            batch_size,
            -1,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        scores = torch.matmul(
            Q,
            K.transpose(-2,-1)
        ) / (self.head_dim ** 0.5)

        attention = torch.softmax(
            scores,
            dim=-1
        )

        attention = self.dropout(attention)

        context = torch.matmul(
            attention,
            V
        )

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.reshape(
            batch_size,
            -1,
            self.hidden_dim
        )

        output = self.out_proj(context)

        return output, attention


# =========================
# Multi-Head Example
# =========================

multi_head = MultiHeadAttention(
    hidden_dim=64,
    num_heads=8
)

mh_output, mh_attention = multi_head(x)

print("\n===== MULTI HEAD =====")
print("Input Shape :", x.shape)
print("Output Shape:", mh_output.shape)
print("Attention Shape:", mh_attention.shape)

# =========================
# Visualize Head 1
# =========================

plt.figure(figsize=(6,5))
plt.imshow(
    mh_attention[0][0].detach().numpy(),
    cmap="Reds"
)
plt.title("Multi-Head Attention (Head 1)")
plt.colorbar()
plt.show()

# =========================
# 3. Encoder Block
# =========================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        hidden_dim,
        num_heads,
        ff_dim=256,
        dropout=0.1
    ):
        super().__init__()

        self.attention = MultiHeadAttention(
            hidden_dim,
            num_heads,
            dropout
        )

        self.norm1 = nn.LayerNorm(hidden_dim)

        self.feedforward = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, hidden_dim)
        )

        self.norm2 = nn.LayerNorm(hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        attention_output, attention_weights = self.attention(x)

        x = self.norm1(
            x + self.dropout(attention_output)
        )

        ff_output = self.feedforward(x)

        x = self.norm2(
            x + self.dropout(ff_output)
        )

        return x, attention_weights


# =========================
# Encoder Test
# =========================

encoder = EncoderBlock(
    hidden_dim=64,
    num_heads=8
)

encoder_output, encoder_attention = encoder(x)

print("\n===== ENCODER BLOCK =====")
print("Input Shape :", x.shape)
print("Output Shape:", encoder_output.shape)

# =========================
# Optional Lightweight Training Example
# =========================

class SimpleClassifier(nn.Module):

    def __init__(
        self,
        hidden_dim=64,
        num_heads=8,
        num_classes=3
    ):
        super().__init__()

        self.encoder = EncoderBlock(
            hidden_dim,
            num_heads
        )

        self.classifier = nn.Linear(
            hidden_dim,
            num_classes
        )

    def forward(self, x):

        x, attn = self.encoder(x)

        pooled = x.mean(dim=1)

        logits = self.classifier(pooled)

        return logits, attn


model = SimpleClassifier()

dummy_inputs = torch.randn(
    16,
    20,
    64
)

dummy_labels = torch.randint(
    0,
    3,
    (16,)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()

for epoch in range(3):

    optimizer.zero_grad()

    logits, _ = model(dummy_inputs)

    loss = criterion(
        logits,
        dummy_labels
    )

    loss.backward()

    optimizer.step()

    predictions = logits.argmax(dim=1)

    accuracy = (
        predictions == dummy_labels
    ).float().mean()

    print(
        f"Epoch {epoch+1} | Loss={loss.item():.4f} | Accuracy={accuracy:.4f}"
    )

# =========================
# Reflection
# =========================

print("\n===== REFLECTION =====")
print("""
1. Single-head attention learns one relationship at a time.

2. Multi-head attention captures different types of relationships simultaneously.

3. Encoder blocks improve representation quality using:
   - Multi-head attention
   - Residual connections
   - Layer normalization
   - Feed-forward networks

4. Pretrained models such as BERT or DistilBERT generally outperform
   lightweight custom transformers because they have been trained on
   massive text corpora.

5. Custom attention models are faster, smaller, and easier to train
   but typically achieve lower accuracy.
""")